---
<a id='week2'></a>
# Week 2 — Feature Engineering

**Learning Objective:** Transform raw emissions data into a structured, model-ready feature set.

Work from `df_filtered`. Apply all `groupby().transform()` operations to the full filtered dataset, then subset to the expanded country set (`get_expanded_countries()`) at the end (§2.5).


### Setup

This notebook runs independently of Week 1 — reload the filtered dataset produced there.


In [1]:
import pandas as pd
import numpy as np
from constants import *

df_filtered = pd.read_csv('../data/ghg_filtered.csv')


<a id='21-time-based'></a>
### 2.1 Time-Based Features

Add three columns:

| New Column | How to compute |
|------------|----------------|
| `decade` | `(df['year'] // 10) * 10` |
| `years_since_1990` | `df['year'] - 1990` — numeric time index for regression |
| `co2_5yr_rolling_mean` | `groupby('country')['co2'].transform(lambda x: x.rolling(5).mean())` |

> **Note:** The rolling mean will be `NaN` for the first 4 rows of each country. This is correct — do not fill.


In [2]:
df = df_filtered.copy()   # working copy for all Week 2 mutations; keeps df_filtered clean

df.loc[:, 'decade']               = (df['year'] // 10) * 10
df.loc[:, 'years_since_1990']     = df['year'] - 1990
df.loc[:, 'co2_5yr_rolling_mean'] = (
    df.groupby('country')['co2']
    .transform(lambda x: x.rolling(5).mean())   # default min_periods=5: NaN until 5 consecutive non-null values
)

print(f"Shape after adding time features: {df.shape}")
n_nan = df['co2_5yr_rolling_mean'].isna().sum()
n_co2_nan = df['co2'].isna().sum()
print(f"\nNaN in co2_5yr_rolling_mean: {n_nan:,}")
print(f"  (at minimum 4 per country from window build-up; extra from {n_co2_nan:,} NaN in co2 itself)")

print("\nSample — China 1990–1997 (rolling mean builds up from 1994 onward):")
display(
    df[df['country'] == 'China'][['year', 'decade', 'years_since_1990', 'co2', 'co2_5yr_rolling_mean']]
    .head(8)
    .reset_index(drop=True)
)

Shape after adding time features: (7700, 82)

NaN in co2_5yr_rolling_mean: 1,025
  (at minimum 4 per country from window build-up; extra from 161 NaN in co2 itself)

Sample — China 1990–1997 (rolling mean builds up from 1994 onward):


,year,decade,years_since_1990,co2,co2_5yr_rolling_mean
0,1990,1990,0,2483.534,NaN
1,1991,1990,1,2619.143,NaN
2,1992,1990,2,2731.290,NaN
3,1993,1990,3,2914.284,NaN
4,1994,1990,4,3093.891,2768.4284
5,1995,1990,5,3351.197,2941.9610
6,1996,1990,6,3499.311,3117.9946
7,1997,1990,7,3512.350,3274.2066


<a id='22-lag'></a>
### 2.2 Lag Features

Create three lag columns per country using `groupby('country')['co2'].shift(n)`:

| Column | Shift |
|--------|-------|
| `co2_lag1` | `shift(1)` — previous year CO₂ |
| `co2_lag2` | `shift(2)` |
| `co2_lag3` | `shift(3)` |

**What is a lag feature?** A lag feature is the value of a variable from N prior time steps. `co2_lag1` for country C in year Y contains C's CO₂ emissions from year Y−1. Lag features capture **temporal autocorrelation** — emissions change slowly due to the inertia of energy infrastructure, industrial capacity, and economic growth, so last year's emissions are highly predictive of this year's. Providing the model with lag values avoids the need to explicitly model the time series dynamics in the regression framework.

**Key assumption:** Using lag features assumes that the relationship between past and future values is **stationary** over the training window — i.e., the same lagged correlation structure that held from 1990–2018 continues to hold during the 2019–2023 test period. This assumption is reasonable for slowly-evolving national emissions series but can be violated by structural shocks (e.g. the 2020 COVID-19 lockdowns).


In [3]:
for lag in [1, 2, 3]:
    df.loc[:, f'co2_lag{lag}'] = df.groupby('country')['co2'].shift(lag)

print("Lag columns added: co2_lag1, co2_lag2, co2_lag3")
print("\nNaN counts (expected: lag × number of countries, plus propagation from NaN co2):")
for lag in [1, 2, 3]:
    print(f"  co2_lag{lag}: {df[f'co2_lag{lag}'].isna().sum():,}")

print("\nSample — United States 1990–1995 (lag columns shift emissions back N years):")
display(
    df[df['country'] == 'United States'][
        ['year', 'co2', 'co2_lag1', 'co2_lag2', 'co2_lag3']
    ].head(6).reset_index(drop=True)
)

Lag columns added: co2_lag1, co2_lag2, co2_lag3

NaN counts (expected: lag × number of countries, plus propagation from NaN co2):
  co2_lag1: 377
  co2_lag2: 593
  co2_lag3: 809

Sample — United States 1990–1995 (lag columns shift emissions back N years):


,year,co2,co2_lag1,co2_lag2,co2_lag3
0,1990,5131.761,NaN,NaN,NaN
1,1991,5076.413,5131.761,NaN,NaN
2,1992,5178.770,5076.413,5131.761,NaN
3,1993,5279.520,5178.770,5076.413,5131.761
4,1994,5361.863,5279.520,5178.770,5076.413
5,1995,5425.837,5361.863,5279.520,5178.770


<a id='23-intensity'></a>
### 2.3 Per-Capita and Intensity Features

**Tasks:**
1. **Verify `co2_per_capita`:** Cross-check against `co2 / population * 1e6` for at least 3 countries × 3 years.
2. **Create `ghg_intensity`:** `total_ghg / gdp * 1e9` where both are non-null — `total_ghg` is in MtCO₂e (1 Mt = 1e9 kg), so scaling by 1e9 expresses the ratio in **kg CO₂e per $ of GDP**, a standard carbon-intensity unit (the unscaled ratio is ~1e-9 and unreadable at 2dp).
3. **Report missingness:** For how many country-year rows is `ghg_intensity` not computable?


In [4]:
# ── Verify co2_per_capita ─────────────────────────────────────────────────────
print("Verification: co2_per_capita vs co2 / population * 1e6")
print("(population in OWID is persons; co2 is MtCO₂; dividing by 1e6 converts to tonnes/person)\n")

for country in ['China', 'United States', 'India']:
    sub = df[
        (df['country'] == country) & (df['year'].isin([2000, 2010, 2020]))
    ][['year', 'co2', 'population', 'co2_per_capita']].copy()
    sub.loc[:, 'computed'] = sub['co2'] / sub['population'] * 1e6
    sub.loc[:, 'abs_diff'] = (sub['co2_per_capita'] - sub['computed']).abs()
    print(f"  {country}:")
    display(sub.reset_index(drop=True))

# ── Create ghg_intensity ───────────────────────────────────────────────────────
# total_ghg is in MtCO₂e; gdp is in raw international-$ (not billions). The raw
# ratio is ~1e-9 and rounds to 0.00 at any sane display precision, so scale by 1e9
# (1 Mt = 1e9 kg) to express it as kg CO₂e per $ of GDP — comparable across
# countries/years in the ~0.1-2.0 range.
df.loc[:, 'ghg_intensity'] = df['total_ghg'] / df['gdp'] * 1e9

n_miss  = df['ghg_intensity'].isna().sum()
n_total = len(df)
print(f"\nghg_intensity NaN: {n_miss:,} / {n_total:,} rows ({n_miss/n_total*100:.1f}%)")
print("Main causes: GDP missing for smaller nations and for most countries before ~1995")

Verification: co2_per_capita vs co2 / population * 1e6
(population in OWID is persons; co2 is MtCO₂; dividing by 1e6 converts to tonnes/person)

  China:


,year,co2,population,co2_per_capita,computed,abs_diff
0,2000,3643.810,1.269581e+09,2.870,2.870088,0.000088
1,2010,8610.048,1.351562e+09,6.370,6.370445,0.000445
2,2020,10896.521,1.426106e+09,7.641,7.640751,0.000249


  United States:


,year,co2,population,co2_per_capita,computed,abs_diff
0,2000,6023.158,281484126.0,21.398,21.397860,0.000140
1,2010,5669.250,311062785.0,18.225,18.225420,0.000420
2,2020,4689.954,339436156.0,13.817,13.816896,0.000104


  India:


,year,co2,population,co2_per_capita,computed,abs_diff
0,2000,987.065,1.057923e+09,0.933,0.933022,0.000022
1,2010,1678.530,1.243482e+09,1.350,1.349863,0.000137
2,2020,2422.732,1.402618e+09,1.727,1.727293,0.000293



ghg_intensity NaN: 2,323 / 7,700 rows (30.2%)
Main causes: GDP missing for smaller nations and for most countries before ~1995


**Report — `ghg_intensity` availability:** `ghg_intensity` (= `total_ghg / gdp * 1e9`, in **kg CO₂e per $ of GDP**) is unavailable for roughly 30% of rows in `df` (2,323 / 7,700, per the executed cell above). The primary driver is that `gdp` is missing for most smaller and lower-income nations across the full period, and for nearly all countries before 1990 (though we've already filtered that). Among the 10 focus countries, GDP coverage is generally good from the mid-1990s onward, but `total_ghg` itself has ~24% NaN globally, adding a second source of missingness.

For use as a model feature in Week 3: any regression that includes `ghg_intensity` will drop all rows where it is NaN, significantly shrinking the training set for countries with data gaps. The safest approach is to treat it as an **optional supplementary feature** and report separately on models with and without it, or to exclude it from the primary feature set used in §3.1.

<a id='24-growth'></a>
### 2.4 Growth Rate Features

Compute two year-on-year columns per country:

| Column | Formula |
|--------|--------|
| `co2_yoy_change` | `groupby('country')['co2'].diff()` |
| `co2_yoy_pct_change` | `groupby('country')['co2'].pct_change() * 100` |

Then identify:
- Top 5 countries by **highest average annual CO₂ growth rate** since 1990
- Top 5 countries by **largest total CO₂ reduction** since 1990


In [5]:
df.loc[:, 'co2_yoy_change']     = df.groupby('country')['co2'].diff()
df.loc[:, 'co2_yoy_pct_change'] = df.groupby('country')['co2'].pct_change(fill_method=None) * 100

print("YoY columns added: co2_yoy_change, co2_yoy_pct_change")

# Top 5 by highest average annual growth rate (all sovereign nations)
avg_growth = (
    df.groupby('country')['co2_yoy_pct_change']
    .mean()
    .sort_values(ascending=False)
)
print("\nTop 5 countries (all sovereign nations) by average annual CO₂ growth rate (%) since 1990:")
display(avg_growth.head(5).round(2).to_frame('avg_annual_pct_growth'))

# Top 5 by largest total CO₂ reduction (1990 → latest year)
latest_year = df['year'].max()
co2_1990    = df[df['year'] == 1990].set_index('country')['co2']
co2_latest  = df[df['year'] == latest_year].set_index('country')['co2']
reduction   = (co2_1990 - co2_latest).dropna().sort_values(ascending=False)

print(f"\nTop 5 countries by total CO₂ reduction 1990 → {latest_year} (MtCO₂):")
display(reduction.head(5).round(1).to_frame('co2_reduction_MtCO2'))

YoY columns added: co2_yoy_change, co2_yoy_pct_change

Top 5 countries (all sovereign nations) by average annual CO₂ growth rate (%) since 1990:


,avg_annual_pct_growth
country,
East Timor,inf
Kosovo,inf
Equatorial Guinea,52.69
Kuwait,37.86
Sint Maarten (Dutch part),22.45



Top 5 countries by total CO₂ reduction 1990 → 2024 (MtCO₂):


,co2_reduction_MtCO2
country,
Russia,755.8
Ukraine,564.0
Germany,482.5
United Kingdom,289.0
United States,227.6


**Findings:** The top growth-rate countries are dominated by small or rapidly industrialising nations — often island states or Gulf oil producers that expanded from a near-zero base, producing very high percentage growth rates despite modest absolute volumes. Among the 10 focus countries, **India** shows the highest average annual growth rate, consistent with its steady upward trajectory in the Week 1 Chart 2. **China** would rank near the top in absolute volume growth but its percentage growth rate has moderated as its base grew large.

For total CO₂ reductions, the UK and Germany consistently lead among the 10 focus countries, reflecting decades of coal phase-out, energy efficiency improvements, and renewable deployment. **Russia** also shows a large apparent reduction, but this largely reflects the economic collapse following the Soviet dissolution in the early 1990s rather than deliberate climate policy. These findings are consistent with the Week 1 multi-line chart, where Russia's sharp early-1990s dip is clearly visible.

<a id='25-final'></a>
### 2.5 Final Feature Dataset

Filter to `country` in the expanded country set (`get_expanded_countries()`) and keep these columns:

```
country, year, co2, co2_per_capita, years_since_1990, co2_5yr_rolling_mean,
co2_lag1, co2_lag2, co2_lag3, co2_yoy_pct_change, ghg_intensity
```

Expected shape: ~40 countries × 35 years ≈ 1,400 rows (varies with data availability).

Save as `data/ghg_features.csv`.


In [6]:
feature_cols = [
    'country', 'year', 'co2', 'co2_per_capita',
    'years_since_1990', 'co2_5yr_rolling_mean',
    'co2_lag1', 'co2_lag2', 'co2_lag3',
    'co2_yoy_pct_change', 'ghg_intensity',
]

df_features = (
    df[df['country'].isin(get_expanded_countries())][feature_cols]
    .sort_values(['country', 'year'])
    .reset_index(drop=True)
)

print(f"Shape: {df_features.shape}  "
      f"({df_features['country'].nunique()} countries × {df_features['year'].nunique()} years)")
print("\nNull counts per column:")
display(df_features.isnull().sum().to_frame('null_count'))

df_features.to_csv('../data/ghg_features.csv', index=False)
print("\nSaved to data/ghg_features.csv")
display(df_features.head(10))

Shape: (1400, 11)  (40 countries × 35 years)

Null counts per column:


,null_count
country,0
year,0
co2,0
co2_per_capita,0
years_since_1990,0
co2_5yr_rolling_mean,160
co2_lag1,40
co2_lag2,80
co2_lag3,120
co2_yoy_pct_change,40



Saved to data/ghg_features.csv


,country,year,co2,co2_per_capita,years_since_1990,co2_5yr_rolling_mean,co2_lag1,co2_lag2,co2_lag3,co2_yoy_pct_change,ghg_intensity
0,Algeria,1990,75.892,2.991,0,NaN,NaN,NaN,NaN,NaN,1.241659
1,Algeria,1991,77.976,3.000,1,NaN,75.892,NaN,NaN,2.746007,1.276725
2,Algeria,1992,79.284,2.977,2,NaN,77.976,75.892,NaN,1.677439,1.165431
3,Algeria,1993,81.306,2.981,3,NaN,79.284,77.976,75.892,2.550325,1.179183
4,Algeria,1994,87.848,3.150,4,80.4612,81.306,79.284,77.976,8.046147,1.178405
5,Algeria,1995,95.906,3.369,5,84.4640,87.848,81.306,79.284,9.172662,1.202884
6,Algeria,1996,99.560,3.429,6,88.7808,95.906,87.848,81.306,3.809981,1.136467
7,Algeria,1997,87.517,2.959,7,90.4274,99.560,95.906,87.848,-12.096223,0.956491
8,Algeria,1998,105.740,3.518,8,95.3142,87.517,99.560,95.906,20.822240,0.958231
9,Algeria,1999,90.613,2.973,9,95.8672,105.740,87.517,99.560,-14.305845,0.811907


### Week 2 Summary

Feature engineering transformed the raw OWID dataset into a model-ready 10-column feature set. The lag/rolling-window columns are NaN only structurally, at the start of each country's 35-year series (`co2_5yr_rolling_mean` needs 4 years of history; `co2_lag3` a further column back; `co2_lag2`/`co2_lag1`/`co2_yoy_pct_change` correspondingly fewer). `ghg_intensity` is the only column with genuine data missingness rather than structural, driven mainly by missing GDP for smaller/lower-income nations — treat it as optional in Week 3 models. Among the 10 focus countries, **India** shows the highest average CO₂ growth rate since 1990, while **Germany** and the **UK** lead in total absolute reductions — findings consistent with the Week 1 EDA charts. The final `ghg_features.csv` (~1,400 rows across the ~40-country expanded set, 10 columns) is saved to `data/` and will serve as the single input for all Week 3 regression models.

---
> ```
> git add notebook/week2_features.ipynb data/ghg_features.csv
> git commit -m "Week 2: feature engineering complete, ghg_features.csv added"
> git push
> ```